# Exploring TensorBoard

Use a Python 3 notebook; a GPU is helpful but optional. In Colab, run cells in order. Locally, open the notebook from its folder and install a compatible PyTorch/torchvision pair first. The setup cell installs TensorBoard if needed and obtains `architectures.py` from the FA26 public repository when it is not available locally. The data cell downloads CIFAR-10.

TensorBoard reads local event files containing metrics and other summaries. PyTorch writes them with `SummaryWriter`; you do not need to install TensorFlow or use W&B for this notebook. The example logs learning curves with `add_scalar` and configuration/final metrics with `add_hparams`.

**Task:** run **five distinct hyperparameter configurations** and submit your plots plus answers to part (a) of the homework. The two-batch example below checks that training/logging work; it does not count as a completed experiment. Complete `run()` and the configuration list, then enable `RUN_EXPERIMENTS`. Use unique run names (for example, include a new experiment prefix) to avoid mixing new trials with old event files.


In [ ]:
# Install only the logging package if it is missing. Colab supplies PyTorch.
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

if importlib.util.find_spec("tensorboard") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorboard"])

# Local notebooks use their neighboring helper. A standalone Colab notebook fetches it.
candidates = [Path("architectures.py"), Path("../code/architectures.py")]
helper = next((p for p in candidates if p.is_file()), Path("architectures.py"))
if not helper.is_file():
    urlretrieve(
        "https://raw.githubusercontent.com/Berkeley-CS182/cs182fa26_public/main/hw06/code/architectures.py",
        helper,
    )
spec = importlib.util.spec_from_file_location("hw06_architectures", helper)
architectures = importlib.util.module_from_spec(spec)
spec.loader.exec_module(architectures)
BasicConvNet, ResNet18, MLP = architectures.BasicConvNet, architectures.ResNet18, architectures.MLP

import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 182
DATA_ROOT = "./data"
NUM_WORKERS = 0  # portable on Colab/Windows and makes sampling straightforward
print("PyTorch:", torch.__version__, "device:", device)


The supplied helpers train a model and evaluate it on a **fixed validation split** drawn from the CIFAR-10 training set. Use validation accuracy to choose hyperparameters. The official test set is not a tuning set. One epoch means one pass over the training split; keep the same number of epochs across configurations, even when batch size changes. Smaller batches produce more optimizer steps per epoch, which is part of the batch-size change.

Every run resets its seed; models with the same architecture start from the same parameters. A seed controls random choices but does not guarantee bit-for-bit agreement across hardware. Training metrics are sample-weighted averages across an epoch, while validation metrics are computed after the epoch in evaluation mode. Accuracy is recorded as a percentage.

`BasicConvNet` is a small CNN; `MLP` is a fully connected network that flattens the image internally. `ResNet18` is an optional, larger CNN with skip connections that add an earlier activation to a later block output. Its code is provided; deriving its architecture is not required. It is initialized without pretrained weights, accepts 32x32 images, and can be slower than the small CNN.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
all_training_data = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True, transform=transform)
indices = torch.randperm(len(all_training_data),
                         generator=torch.Generator().manual_seed(SEED)).tolist()
split_at = int(0.9 * len(indices))
trainset = Subset(all_training_data, indices[:split_at])
valset = Subset(all_training_data, indices[split_at:])
print(f"Training: {len(trainset)}, validation: {len(valset)} examples")
# The official CIFAR-10 test set is reserved for a final evaluation after tuning.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def prepare_run(config):
    seed_everything(config["seed"])
    model_factories = {"CNN": BasicConvNet, "MLP": MLP, "ResNet18": ResNet18}
    model = model_factories[config["architecture"]]().to(device)
    if config["optimizer"] == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=config["learning_rate"],
                                    momentum=config["momentum"])
    elif config["optimizer"] == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    else:
        raise ValueError("Choose SGD or Adam")
    trainloader = DataLoader(trainset, batch_size=config["batch_size"], shuffle=True,
                            num_workers=NUM_WORKERS,
                            generator=torch.Generator().manual_seed(config["seed"]))
    valloader = DataLoader(valset, batch_size=config["batch_size"], shuffle=False,
                          num_workers=NUM_WORKERS,
                          generator=torch.Generator().manual_seed(config["seed"]))
    return model, optimizer, trainloader, valloader

def train_one_epoch(model, optimizer, loader, max_batches=None):
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for batch_index, (inputs, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)  # each architecture handles flattening internally
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)
    if count == 0:
        raise ValueError("No training examples were processed")
    return {"train/loss": loss_sum / count, "train/accuracy": 100 * correct / count}

@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    for batch_index, (inputs, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        logits = model(inputs)
        loss_sum += nn.functional.cross_entropy(logits, labels, reduction="sum").item()
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)
    if count == 0:
        raise ValueError("No validation examples were processed")
    return {"val/loss": loss_sum / count, "val/accuracy": 100 * correct / count}

base_config = dict(learning_rate=0.01, batch_size=128, architecture="CNN",
                   optimizer="SGD", momentum=0.9, epochs=2, seed=SEED)


## Runnable logging example

The example logs one epoch on just two training and two validation batches.

In [ ]:
def tensorboard_example(config, run_name="example", max_batches=2):
    # A bounded demonstration. Use max_batches=None for a full experiment.
    model, optimizer, trainloader, valloader = prepare_run(config)
    with SummaryWriter(log_dir=f"runs/hw06/{run_name}") as writer:
        for epoch in range(1, config["epochs"] + 1):
            metrics = train_one_epoch(model, optimizer, trainloader, max_batches)
            metrics.update(evaluate(model, valloader, max_batches))
            for name, value in metrics.items():
                writer.add_scalar(name, value, epoch)
            print(f"{run_name}, epoch {epoch}: {metrics}")
        # Distinct tags keep final HParams metrics separate from the scalar curves.
        writer.add_hparams(config, {"hparam/val_accuracy": metrics["val/accuracy"],
                                   "hparam/val_loss": metrics["val/loss"]},
                           run_name="hparams")
    return metrics

example_config = dict(base_config, epochs=1)
example_metrics = tensorboard_example(example_config)
print("Example processed only two batches; do not report it as a full experiment.")


## Inspecting TensorBoard

After running the example, execute the next cell to open the dashboard. The Scalars tab shows learning curves; the HParams tab compares logged hyperparameters and final metrics. **TensorBoard can filter and sort by configurations in HParams**; it is not limited to run-name regular expressions. The [official HParams guide](https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams) illustrates the table, parallel-coordinate, and scatter-plot views.

If the notebook extension is unavailable locally, run `tensorboard --logdir runs/hw06` in a terminal and open its printed local URL. For repeated experiments, use a fresh run-name prefix rather than overwriting logs.


In [ ]:
# Run this cell interactively when you want to view the dashboard.
OPEN_DASHBOARD = False
if OPEN_DASHBOARD:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", "--logdir runs/hw06")


## Your experiments

Choose distinct configurations, use the full training/validation data, and record every changed setting. If you want to attribute an effect to one setting (an ablation), hold the other settings fixed. Report the best configuration within your actual search, not a guessed optimum.

In [ ]:
hyperparameters = []  # TODO: choose five distinct configuration dictionaries.

def run(config, run_name, max_batches=None):
    # TODO: adapt the example to train, validate, and log one configuration.
    # Return a dictionary containing val/accuracy and val/loss.
    raise NotImplementedError("Implement your TensorBoard experiment")

RUN_EXPERIMENTS = False  # Set True after completing run() and the five configs.
if RUN_EXPERIMENTS:
    assert len(hyperparameters) == 5, "Choose five configurations"
    experiment_results = [run(config, run_name=f"trial-{i}")
                          for i, config in enumerate(hyperparameters)]
